## 前言

在这个全民 agent 的时代，我们如何快速构建一个 agent 呢？langchain作为一个通用agent框架，提供了丰富的组件，使得我们能够快速构建一个agent。



## 前置条件

- 安装 Python 3.13
- 安装 uv：项目管理
- 获取 LLM api-key

## 初始化项目

进入项目目录，执行如下命令：
- 初始化项目：```uv init```

## 安装依赖包
- 安装依赖包：```uv add langchain langchain-openai```



## 创建一个基础agent

agent = llm + tools，所以我们接下来的步骤就非常简单：
1. 初始化llm
2. 添加tools
3. 创建agent
4. 运行agent

### 步骤一：初始化llm

In [42]:
import os
from langchain.chat_models import init_chat_model

llm = init_chat_model(
    model="Qwen/Qwen3-8B",
    model_provider="openai",
    api_key=os.environ["SILICONFLOW_API_KEY"],
    base_url="https://api.siliconflow.cn/v1",
)

### 步骤二：添加tools

In [43]:
def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

### 步骤三：创建agent

In [44]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[get_weather],
    system_prompt="You are a helpful assistant",
)


### 步骤四：执行agent


In [45]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's the weather in San Francisco?"}]}
)

for message in result["messages"]:
    message.pretty_print()


================================ Human Message =================================

What's the weather in San Francisco?
================================== Ai Message ==================================
Tool Calls:
  get_weather (019f1659398ae5e7b243aecb46af1fd9)
 Call ID: 019f1659398ae5e7b243aecb46af1fd9
  Args:
    city: San Francisco
================================= Tool Message =================================
Name: get_weather

It's always sunny in San Francisco!
================================== Ai Message ==================================

The weather in San Francisco is currently sunny! ☀️ Whether you're planning a beach day or a stroll through the city, the sunshine is sure to brighten your mood. Let me know if you need more details!


## 打造一个现实世界的agent

### 步骤一：定义 system_prompt

In [ ]:
SYSTEM_PROMPT = """You are a literary data assistant.

## Capabilities

- `fetch_text_from_url`: loads document text from a URL into the conversation.
Do not guess line counts or positions—ground them in tool results from the saved file."""

### 步骤二：创建tools

以下创建了一个fetch_text_from_url函数，用于从指定URL获取文本数据。

In [ ]:
import urllib.error
import urllib.request

from langchain.tools import tool


@tool
def fetch_text_from_url(url: str) -> str:
    """Fetch the document from a URL.
    """
    req = urllib.request.Request(
        url,
        headers={"User-Agent": "Mozilla/5.0 (compatible; quickstart-research/1.0)"},
    )
    try:
        with urllib.request.urlopen(req, timeout=120) as resp:
            raw = resp.read()
    except urllib.error.URLError as e:
        return f"Fetch failed: {e}"
    text = raw.decode("utf-8", errors="replace")
    return text

### 步骤三：配置模型

In [ ]:
import os
from langchain.chat_models import init_chat_model

llm = init_chat_model(
    model="Qwen/Qwen3-8B",
    model_provider="openai",
    api_key=os.environ["SILICONFLOW_API_KEY"],
    base_url="https://api.siliconflow.cn/v1",
)


### 步骤四：添加memory

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

### 步骤五：创建agent

#### 创建 langchain agent

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[fetch_text_from_url],
    system_prompt=SYSTEM_PROMPT,
    checkpointer=checkpointer,
)

#### 创建 deepagents

In [ ]:
from deepagents import create_deep_agent

deep_agent = create_deep_agent(
    model=llm,
    tools=[fetch_text_from_url],
    system_prompt=SYSTEM_PROMPT,
    checkpointer=checkpointer,
)

### 步骤六：执行agent

#### 执行 langchain agent

In [ ]:
content = f"""Project Gutenberg hosts a full plain-text copy of F. Scott Fitzgerald's The Great Gatsby.
URL: https://www.gutenberg.org/files/64317/64317-0.txt

Answer as much as you can:

1) How many lines in the complete Gutenberg file contain the substring `Gatsby` (count lines, not occurrences within a line, each line ends with a line break).
2) The 1-based line number of the first line in the file that contains `Daisy`.
3) A two-sentence neutral synopsis.

Do your best on (1) and (2). If at any point you realize you cannot **verify** an exact answer with
your available tools and reasoning, do not fabricate numbers: use `null` for that field and spell out
the limitation in `how_you_computed_counts`. If you encounter any errors please report what the error was and what the error message was."""


agent_result = agent.invoke(
    {"messages": [{"role": "user", "content": content}]},
    config={"configurable": {"thread_id": "great-gatsby-lc"}},
)

print(agent_result["messages"][-1].content_blocks)

print('-' * 180)
for message in agent_result["messages"]:
    message.pretty_print()


#### 执行 deepagent

In [ ]:
content = f"""Project Gutenberg hosts a full plain-text copy of F. Scott Fitzgerald's The Great Gatsby.
URL: https://www.gutenberg.org/files/64317/64317-0.txt

Answer as much as you can:

1) How many lines in the complete Gutenberg file contain the substring `Gatsby` (count lines, not occurrences within a line, each line ends with a line break).
2) The 1-based line number of the first line in the file that contains `Daisy`.
3) A two-sentence neutral synopsis.

Do your best on (1) and (2). If at any point you realize you cannot **verify** an exact answer with
your available tools and reasoning, do not fabricate numbers: use `null` for that field and spell out
the limitation in `how_you_computed_counts`. If you encounter any errors please report what the error was and what the error message was."""

deep_agent_result = deep_agent.invoke(
    {"messages": [{"role": "user", "content": content}]},
    config={"configurable": {"thread_id": "great-gatsby-da"}},
)

print(deep_agent_result["messages"][-1].content_blocks)

print('-' * 180)
for message in deep_agent_result["messages"]:
    message.pretty_print()